# 日経新聞の記事検索結果を取得する

日経電子版で「熊本地震」を検索し、2026年7月27日以降に公開された記事のタイトル・本文・公開日時・URLを取得します。

- 取得するのは、未ログイン状態で各記事ページから閲覧できる本文だけです。有料部分やログイン必須部分は取得しません。
- 短時間に大量アクセスしないよう、待機時間と取得件数の上限を設けています。
- 実行前に日経電子版の利用規約・著作権・robots.txtを確認し、取得データは許可された範囲で利用してください。


In [8]:
# 初回だけ実行してください
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "requests", "beautifulsoup4", "pandas", "playwright"
])



[notice] A new release of pip is available: 24.3.1 -> 26.2
[notice] To update, run: pip install --upgrade pip


0

In [9]:
import json
import time
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://www.nikkei.com"
SEARCH_URL = f"{BASE_URL}/search"
KEYWORD = "熊本地震"
START_DATE = pd.Timestamp("2026-07-27", tz="Asia/Tokyo")

# 日経の検索結果は通常1ページ10件です。
# 検索画面に表示された合計件数まで自動取得します。
REQUEST_INTERVAL = 1.5  # 検索ページ・各記事へのアクセス間隔（秒）
TIMEOUT = 20

session = requests.Session()
session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/126.0 Safari/537.36"
    ),
    "Accept-Language": "ja,en-US;q=0.8,en;q=0.6",
})


## ログイン情報の入力

IDとパスワードは変数にだけ保持され、ノートブックファイルには保存されません。パスワードは入力中も画面に表示されません。


In [10]:
from getpass import getpass

NIKKEI_ID = input("日経ID（メールアドレス）: ").strip()
NIKKEI_PASSWORD = getpass("パスワード: ")


In [11]:
from playwright.async_api import async_playwright


async def login_to_nikkei(requests_session, nikkei_id, password):
    """実ブラウザで日経にログインし、認証Cookieをrequests.Sessionへ渡す。"""
    if not nikkei_id or not password:
        raise ValueError("日経IDとパスワードを入力してください。")

    async with async_playwright() as playwright:
        # PCにインストール済みのGoogle Chromeを利用するため、
        # playwright install chromium は通常不要です。
        browser = await playwright.chromium.launch(channel="chrome", headless=False)
        context = await browser.new_context(locale="ja-JP")
        page = await context.new_page()
        await page.goto(f"{BASE_URL}/login", wait_until="domcontentloaded", timeout=60_000)

        username = page.locator(
            'input[autocomplete="username"], input[type="email"], '
            'input[name*="mail" i], input[type="text"]'
        ).first
        await username.wait_for(state="visible", timeout=30_000)
        await username.fill(nikkei_id)

        password_input = page.locator('input[type="password"]').first
        if not await password_input.is_visible():
            # ID入力とパスワード入力が2画面に分かれている場合
            await page.locator('button[type="submit"]').first.click()
            await password_input.wait_for(state="visible", timeout=30_000)

        await password_input.fill(password)
        await page.locator('button[type="submit"]').first.click()

        print("ブラウザでログインを確認してください。追加認証が表示された場合は完了してください。")
        input("ログイン完了後、このセルの入力欄でEnterを押してください: ")

        cookies = await context.cookies(BASE_URL)
        for cookie in cookies:
            requests_session.cookies.set(
                cookie["name"],
                cookie["value"],
                domain=cookie.get("domain"),
                path=cookie.get("path", "/"),
            )
        await browser.close()

    # requests側でもログイン状態を確認する。
    response = requests_session.get(BASE_URL, timeout=TIMEOUT)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    html = soup.find("html")
    user_rank = html.get("data-user-rank") if html else None
    if user_rank == "anonymous":
        raise RuntimeError("ログイン状態を確認できませんでした。もう一度実行してください。")

    print(f"ログインしました（user rank: {user_rank or 'authenticated'}）。")


await login_to_nikkei(session, NIKKEI_ID, NIKKEI_PASSWORD)

# 認証情報そのものは以降不要なので、メモリ上から削除します。
del NIKKEI_ID, NIKKEI_PASSWORD


ブラウザでログインを確認してください。追加認証が表示された場合は完了してください。
ログインしました（user rank: authenticated）。


In [12]:
def get_soup(url, *, params=None):
    """URLを取得してBeautifulSoupを返す。HTTPエラー時は例外にする。"""
    response = session.get(url, params=params, timeout=TIMEOUT)
    response.raise_for_status()
    response.encoding = response.apparent_encoding or "utf-8"
    return BeautifulSoup(response.text, "html.parser")


def parse_search_page(soup):
    """検索結果1ページの記事、合計件数、次の取得条件を返す。"""
    rows = []
    for item in soup.select('[data-search-result-item="search"]'):
        link = item.select_one('a[href*="/article/"]')
        if link is None:
            # 写真記事など、絶対URLで掲載される検索結果にも対応
            link = item.select_one('a[href*="nikkei.com/"]')
        if link is None:
            continue

        url = urljoin(BASE_URL, link.get("href", ""))
        title = link.get("title") or link.get_text(" ", strip=True)
        time_tag = item.select_one("time[datetime]")
        rows.append({
            "title": title.strip(),
            "published_at": time_tag.get("datetime") if time_tag else None,
            "url": url,
        })

    total_node = soup.select_one("[data-search-total-count]")
    total_count = None
    if total_node:
        count_text = total_node.get_text(strip=True).replace(",", "")
        if count_text.isdigit():
            total_count = int(count_text)

    next_node = soup.select_one("[data-rn-partial-stream-next]")
    next_params = None
    if next_node:
        try:
            next_params = json.loads(next_node["data-rn-partial-stream-next"])
        except (json.JSONDecodeError, KeyError):
            pass

    return rows, total_count, next_params


def search_articles(keyword, start_date):
    """「もっと見る」の取得条件を追い、指定日以降の結果を取得する。"""
    found = []
    seen = set()
    params = {"keyword": keyword}
    total_count = None
    page = 0

    while True:
        page += 1
        soup = get_soup(SEARCH_URL, params=params)
        page_rows, page_total, next_params = parse_search_page(soup)
        if total_count is None and page_total is not None:
            total_count = page_total
            print(f"検索結果の合計件数: {total_count:,} 件")
        new_count = 0
        reached_older_articles = False

        for row in page_rows:
            if row["url"] in seen:
                continue
            seen.add(row["url"])
            new_count += 1
            published_at = pd.to_datetime(row["published_at"], errors="coerce")
            if pd.isna(published_at):
                continue
            if published_at.tzinfo is None:
                published_at = published_at.tz_localize(start_date.tz)
            else:
                published_at = published_at.tz_convert(start_date.tz)
            if published_at < start_date:
                reached_older_articles = True
                continue
            found.append(row)

        print(f"検索一覧を取得中: {start_date.date()} 以降 {len(found):,} 件")

        if reached_older_articles:
            print(f"{start_date.date()} より前の記事に達したため検索を終了します。")
            break
        if total_count is not None and len(found) >= total_count:
            break
        if not page_rows or new_count == 0 or not next_params:
            if total_count is not None and len(found) < total_count:
                print(
                    f"警告: 合計 {total_count:,} 件のうち {len(found):,} 件で"
                    "次の検索結果がなくなりました。"
                )
            break

        params = next_params
        time.sleep(REQUEST_INTERVAL)

    return found


def news_article_json_ld(soup):
    """JSON-LDからNewsArticleメタデータを探す。"""
    for script in soup.select('script[type="application/ld+json"]'):
        try:
            data = json.loads(script.string or script.get_text())
        except (json.JSONDecodeError, TypeError):
            continue

        candidates = data.get("@graph", []) if isinstance(data, dict) else data
        if isinstance(candidates, dict):
            candidates = [candidates]
        if isinstance(data, dict):
            candidates = [data, *candidates]

        for candidate in candidates if isinstance(candidates, list) else []:
            if isinstance(candidate, dict) and candidate.get("@type") in {
                "NewsArticle", "Article"
            }:
                return candidate
    return {}


def parse_article_page(soup, search_row):
    """記事ページからタイトル・本文・公開日時を取り出す。"""
    metadata = news_article_json_ld(soup)

    # 現行ページの本文段落は paragraph_ で始まるCSS classを持つ。
    # class末尾はビルドごとに変わるため、完全一致にはしない。
    article = soup.find("article")
    scope = article or soup
    paragraph_nodes = scope.select('[class^="paragraph_"]')

    paragraphs = []
    seen_text = set()
    for node in paragraph_nodes:
        text = node.get_text(" ", strip=True)
        if text and text not in seen_text:
            seen_text.add(text)
            paragraphs.append(text)

    body = "\n\n".join(paragraphs)
    body_is_excerpt = False
    if not body:
        # 本文段落が配信されていない場合は、公開メタデータの概要だけを返す。
        description = soup.select_one('meta[name="description"]')
        body = description.get("content", "").strip() if description else ""
        body_is_excerpt = bool(body)

    return {
        "title": metadata.get("headline") or search_row["title"],
        "body": body,
        "published_at": (
            metadata.get("datePublished") or search_row["published_at"]
        ),
        "url": search_row["url"],
        "body_is_excerpt": body_is_excerpt,
    }


def collect_articles(keyword, start_date):
    """検索から記事詳細の取得までを実行する。個別エラーは記録して続行する。"""
    search_rows = search_articles(keyword, start_date)
    print(f"検索結果から {len(search_rows)} 件のURLを取得しました。")

    articles = []
    for index, row in enumerate(search_rows, start=1):
        print(f"[{index}/{len(search_rows)}] {row['title']}")
        try:
            soup = get_soup(row["url"])
            articles.append(parse_article_page(soup, row))
        except requests.RequestException as exc:
            articles.append({
                **row,
                "body": "",
                "body_is_excerpt": False,
                "error": str(exc),
            })
        if index < len(search_rows):
            time.sleep(REQUEST_INTERVAL)

    return articles


In [13]:
articles = collect_articles(KEYWORD, START_DATE)

df = pd.DataFrame(articles)
if not df.empty:
    df["published_at"] = pd.to_datetime(df["published_at"], errors="coerce")
    df = df.sort_values("published_at", ascending=False, na_position="last")
df = df.reset_index(drop=True)
df.insert(0, "ID", range(1, len(df) + 1))

df


検索結果の合計件数: 3,906 件
検索一覧を取得中: 10 / 3906 件
検索一覧を取得中: 20 / 3906 件
検索一覧を取得中: 30 / 3906 件
検索一覧を取得中: 40 / 3906 件
検索一覧を取得中: 50 / 3906 件
検索一覧を取得中: 60 / 3906 件
検索一覧を取得中: 70 / 3906 件
検索一覧を取得中: 80 / 3906 件
検索一覧を取得中: 90 / 3906 件
検索一覧を取得中: 100 / 3906 件
検索一覧を取得中: 110 / 3906 件
検索一覧を取得中: 120 / 3906 件
検索一覧を取得中: 130 / 3906 件
検索一覧を取得中: 140 / 3906 件
検索一覧を取得中: 150 / 3906 件
検索一覧を取得中: 160 / 3906 件
検索一覧を取得中: 170 / 3906 件
検索一覧を取得中: 180 / 3906 件
検索一覧を取得中: 190 / 3906 件
検索一覧を取得中: 200 / 3906 件
検索一覧を取得中: 210 / 3906 件
検索一覧を取得中: 220 / 3906 件
検索一覧を取得中: 230 / 3906 件
検索一覧を取得中: 240 / 3906 件
検索一覧を取得中: 250 / 3906 件
検索一覧を取得中: 260 / 3906 件
検索一覧を取得中: 270 / 3906 件
検索一覧を取得中: 280 / 3906 件
検索一覧を取得中: 290 / 3906 件
検索一覧を取得中: 300 / 3906 件
検索一覧を取得中: 310 / 3906 件
検索一覧を取得中: 320 / 3906 件
検索一覧を取得中: 330 / 3906 件
検索一覧を取得中: 340 / 3906 件
検索一覧を取得中: 350 / 3906 件
検索一覧を取得中: 360 / 3906 件
検索一覧を取得中: 370 / 3906 件
検索一覧を取得中: 380 / 3906 件
検索一覧を取得中: 390 / 3906 件
検索一覧を取得中: 400 / 3906 件
検索一覧を取得中: 410 / 3906 件
検索一覧を取得中: 420 / 3906 件
検索一覧を取得中: 430 / 3906 件
検

,ID,title,body,published_at,url,body_is_excerpt
0,1,プロ野球:オールスター第2戦、全セが乱打戦制す MVPは2本塁打の細川成也,プロ野球のマイナビオールスターゲーム2026は29日、富山市民球場で第2戦が行われ、全セが8...,2026-07-29 22:20:51+09:00,https://www.nikkei.com/article/DGXZQOKC29BP50Z...,False
1,2,北陸電力の26年4〜6月期、純利益21%減 燃料高で期ずれ差損大きく,北陸電力 が29日に発表した2026年4〜6月期の連結決算は、純利益が前年同期比21%減の2...,2026-07-29 20:00:00+09:00,https://www.nikkei.com/article/DGXZQOCC2358X0T...,False
2,3,写真と映像で追う熊本地震 イオンモール爆発・傾く住宅・橋や石垣崩落,,2026-07-29 19:21:44+09:00,https://www.nikkei.com/photo/news/article/?ng=...,False
3,4,防災拠点とは 一時避難や物資輸送の機能、官民で整備進む,▼防災拠点 災害時に活用される防災拠点の一つに、主に都道府県が地域防災計画に位置づける「広域...,2026-07-29 14:00:00+09:00,https://www.nikkei.com/article/DGXZQOUD293590Z...,False
4,5,高市早苗首相「熊本地震、関連調査中含め死者13人に」,高市早苗首相は29日、熊本県で震度7を観測した地震による死者が午前8時30分時点で調査中も含...,2026-07-29 10:36:06+09:00,https://www.nikkei.com/article/DGXZQOUA291320Z...,False
...,...,...,...,...,...,...
3901,3902,避難、欠かせぬ健康管理 血栓防止へ水分を,過去の地震災害では、車内や避難所など窮屈な場所で長期間過ごす被災者が、静脈にできた血栓が肺の...,2011-03-16 10:01:44+09:00,https://www.nikkei.com/article/DGXNZO25131480W...,False
3902,3903,石川県や輪島市の工事で談合容疑 公取委30社に立ち入り,石川県や輪島市が発注する土木工事の入札を巡り地元の建設業者約60社が談合を繰り返していた疑い...,2010-07-14 11:43:39+09:00,https://www.nikkei.com/article/DGXNASDG14019_U...,False
3903,3904,自治体などの震度計、55カ所に不備 気象庁,気象庁は30日、地方自治体や防災科学技術研究所が設置している全国の震度計のうち、29都道府県...,2010-03-30 19:18:27+09:00,https://www.nikkei.com/article/DGXNASDG30043_Q...,False
3904,3905,湯の町「北陸」、格安旅館進出に動揺,関西の奥座敷として栄えてきた北陸地方の温泉地が低料金を売りにする外部資本の進出に揺れている。...,2010-03-28 20:36:20+09:00,https://www.nikkei.com/article/DGXNZO04724800X...,False


In [14]:
# 必要ならCSVへ保存（Excelで開く場合にも文字化けしにくいUTF-8 BOM付き）
output_path = Path("nikkei_熊本地震_20260727以降.csv")
df.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"保存しました: {output_path.resolve()}")


保存しました: /Users/tj/IdeaProjects/sample/get-news/nikkei_能登半島地震.csv
